In [ ]:
!pip install transformers torch

In [ ]:
!pip install datasets tokenizers

In [ ]:
!pip install streamlit

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer,pipeline
from datasets import Dataset, load_dataset
import torch

In [ ]:
import json
with open("train-v1.1.json", "r") as f:
    sdata = json.load(f)


In [ ]:
data = []

for article in sdata["data"]:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            question = qa["question"]
            answer = qa["answers"][0]["text"]
            data.append({"context": context, "question": question, "answer": answer})

import pandas as pd
df = pd.DataFrame(data)
print(df.head())


In [ ]:
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)
sample = df.iloc[0]
context = sample["context"]
question = sample["question"]
result = qa_pipeline(context=context, question=question)

print(f"Question: {question}")
print(f"Predicted Answer: {result['answer']}")
print(f"True Answer: {sample['answer']}")

In [ ]:
preds = []
for i in range (len(df)):
    row = df.iloc[i]
    result = qa_pipeline(context=row["context"], question=row["question"])
    preds.append({
        'questions' : row['question'],
        'predicted_answer' : result['answer'],
        'true_answer' : row['answer']
    })
pd.DataFrame(preds)

In [ ]:
import string
def normalize_text(text):
    text = text.lower()
    text = "".join(char for char in text if char not in string.punctuation)
    return text

def exact_match(pred,true):
    return int(normalize_text(pred) == normalize_text(true))

def f1_score(pred,true):
    pred_tokens = normalize_text(pred).split()
    true_tokens = normalize_text(true).split()

    common = set(pred_tokens) & set(true_tokens)
    if not common:
        return 0.0

    pre = len(common) / len(pred_tokens)
    rec = len(common) / len(true_tokens)

    F1_score = 2 * (pre * rec) / (pre + rec)


In [ ]:
qa_pipe = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    tokenizer="distilbert-base-cased")
exact_match,F1 = [],[]

for i in range (len(df)):
    row = df.iloc[i]
    result = qa_pipeline(context=row["context"], question=row["question"])
    pred = result['answer']
    true = row['answer']

exact_match.append(exact_match(pred,true))
F1.append(F1(pred,true))

print("Exact Match :",sum(exact_match)/len(exact_match))
print("F1 Score :",sum(F1)/len(F1))

In [ ]:
def evaluate_model(model_name,df):
  print("Evaluate model",{model_name})
  qa_pipeline = pipline("Question-Answering",model = model_name)

exact_match,F1 = [], []
for i in range(df):
  row = df.iloc[i]
  result = qa_pipeline(context = row['context'],question = row['question'])
  pred = result['answer']
  true = row['answer']

  exact_match.append(exact_match(pred,true))
  F1.append(F1(pred,true))

em = np.mean(exact_match)
f1 = np.mean(F1)

print("Exact Match :",em)
print("F1 Score :",f1)
return em,F1

In [ ]:
results = {}

models = ["distilbert-base-uncased-distilled-squad",
    "distilbert-base-uncased-distilled-squad",
    "deepset/roberta-base-squad2"
]
for models in model :
  em,F1 = evaluate_model(model,df)
  results[model] = {"Exact Match":em,"F1 Score":F1}

results_df = pd.DataFrame(results).T
print(results_df)

In [ ]:
st.title("Question Answering App using tranformer" )
model_choice = st.select_box("Choose a QA Model:",
    [
        "distilbert-base-uncased-distilled-squad",
        "distilbert-base-uncased-distilled-squad",
        "deepset/roberta-base-squad2"
    ]
)
def load_model(model_name):
return pipeline("question-answering",model = model_name, device=-1)
qa_pipeline = load_model(model_choice)

context = st.text_area("Enter the context:")
question = st.text_input("Enter the question:")

if st.button("Get Answer"):
  if context and question:
    result = qa_pipeline(context=context, question=question)
    st.write("Answer:", result["answer"])
    st.subheader('Amswer')
    st.write(f"Confidence: {result['score']:.2f}")
  else:
    st.warning("Please enter both context and question!")

